In [0]:
from weather_client import WeatherClient

client = WeatherClient()

docs = client.fetch_locations(
    [
        "Chicago, IL",
        "Austin, TX"
    ],
    limit=10
)

print("Documents returned:", len(docs))

for doc in docs[:5]:
    print("LOCATION:", doc["location"])
    print("TYPE:", doc["source_type"])
    print("HEADLINE:", doc["headline"])
    print("TEXT:", doc["narrative_text"][:200])
    print("---")

In [0]:
python_code = open("app.py").read()

import ast
ast.parse(python_code)

print("Syntax OK")

In [0]:
%pip install -r requirements.txt

In [0]:
!python -c "import flask; print('flask ok')"

In [0]:
%pip uninstall -y psycopg2-binary

In [0]:
%pip install psycopg2

In [0]:
%cd /Workspace/Users/owaisferozjafer@gmail.com/Vector-Weather-Retrieval-Service

In [0]:
import flask
import requests
import lakebase
import weather_client

print("all imports OK")

In [0]:
!python -m py_compile app.py

In [0]:
%cd /Workspace/Users/owaisferozjafer@gmail.com/Vector-Weather-Retrieval-Service

import flask
import requests
import lakebase
import weather_client

print("all imports OK")

In [0]:
import lakebase

with lakebase.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT 1;")
        print(cur.fetchone())

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

secret = w.secrets.get_secret(
    scope="database",
    key="lakebase-url"
)

print("secret exists:", secret.value is not None)
print("secret length:", len(secret.value))

In [0]:
import base64
import lakebase

url = base64.b64decode(
    w.secrets.get_secret(
        scope="database",
        key="lakebase-url"
    ).value
).decode("utf-8")

print(url.split("@")[0] + "@<hidden>")

In [0]:
from databricks.sdk import WorkspaceClient
import base64

w = WorkspaceClient()

encoded = w.secrets.get_secret(
    scope="database",
    key="lakebase-url"
).value

url = base64.b64decode(encoded).decode()

print(url.split("@")[0])

In [0]:
import lakebase

with lakebase.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT 1;")
        print(cur.fetchone())

In [0]:
import lakebase

with lakebase.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_name LIKE 'weather%';
        """)
        print(cur.fetchall())

In [0]:
from weather_client import WeatherClient

client = WeatherClient()

docs = client.fetch_locations(
    ["Chicago, IL"],
    limit=3
)

print("Documents:", len(docs))

for d in docs:
    print("---")
    print(d["location"])
    print(d["source_type"])
    print(d["headline"])

In [0]:
import json
import lakebase
from weather_client import WeatherClient

docs = WeatherClient().fetch_locations(
    ["Chicago, IL"],
    limit=3
)

with lakebase.get_connection() as conn:
    with conn.cursor() as cur:
        for doc in docs:
            cur.execute(
                """
                INSERT INTO weather_documents
                (
                    id,
                    location,
                    source_type,
                    headline,
                    narrative_text,
                    issued_at,
                    payload,
                    synced_at
                )
                VALUES
                (
                    %s,%s,%s,%s,%s,%s,%s,now()
                )
                ON CONFLICT(id)
                DO UPDATE SET
                    narrative_text = EXCLUDED.narrative_text,
                    payload = EXCLUDED.payload,
                    synced_at = EXCLUDED.synced_at
                """,
                (
                    doc["id"],
                    doc["location"],
                    doc["source_type"],
                    doc["headline"],
                    doc["narrative_text"],
                    doc["issued_at"],
                    json.dumps(doc["payload"])
                )
            )

    conn.commit()

print("Inserted", len(docs), "documents")

In [0]:
import lakebase

rows = lakebase.run_query(
    """
    SELECT 
        location,
        source_type,
        headline,
        narrative_text
    FROM weather_documents
    ORDER BY synced_at DESC
    LIMIT 5
    """
)

for row in rows:
    print("---")
    print(row["location"])
    print(row["source_type"])
    print(row["headline"])
    print(row["narrative_text"][:100])

In [0]:
from app import app

client = app.test_client()

response = client.post(
    "/weather/sync",
    json={
        "locations": ["Chicago, IL"],
        "limit": 5
    }
)

print(response.status_code)
print(response.json)

In [0]:
import lakebase

lakebase.run_write("""
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS weather_embeddings (
    id BIGSERIAL PRIMARY KEY,
    document_id TEXT NOT NULL,
    chunk_index INTEGER NOT NULL,
    chunk_text TEXT NOT NULL,
    embedding VECTOR(384) NOT NULL,
    model_name TEXT NOT NULL DEFAULT 'sentence-transformers/all-MiniLM-L6-v2',
    created_at TIMESTAMPTZ DEFAULT now(),

    CONSTRAINT weather_embeddings_document_chunk_unique
        UNIQUE(document_id, chunk_index),

    CONSTRAINT weather_embeddings_document_fk
        FOREIGN KEY(document_id)
        REFERENCES weather_documents(id)
);

CREATE INDEX IF NOT EXISTS weather_embeddings_embedding_idx
ON weather_embeddings
USING hnsw (embedding vector_cosine_ops);
""")

print("weather_embeddings table ready")

In [0]:
%cd /Workspace/Users/owaisferozjafer@gmail.com/Vector-Weather-Retrieval-Service

import os
print(os.listdir())

In [0]:
import os

file_path = "/Workspace/Users/owaisferozjafer@gmail.com/Vector-Weather-Retrieval-Service/ingest_weather_embeddings.py"

print(os.path.exists(file_path))

In [0]:
import importlib.util

path = "/Workspace/Users/owaisferozjafer@gmail.com/Vector-Weather-Retrieval-Service/ingest_weather_embeddings.py"

spec = importlib.util.spec_from_file_location(
    "ingest_weather_embeddings",
    path
)

module = importlib.util.module_from_spec(spec)

spec.loader.exec_module(module)

print("loaded successfully")

In [0]:
import importlib

module = importlib.import_module("ingest_weather_embeddings")

print("Loaded:", module)

In [0]:
module.main()

In [0]:
import lakebase

rows = lakebase.run_query(
    """
    SELECT
        document_id,
        chunk_index,
        LEFT(chunk_text, 80) AS preview,
        model_name,
        vector_dims(embedding) AS dimensions
    FROM weather_embeddings
    ORDER BY created_at DESC
    LIMIT 10
    """
)

for r in rows:
    print(r)

In [0]:
import app
print("app import OK")

In [0]:
%cd /Workspace/Users/owaisferozjafer@gmail.com/Vector-Weather-Retrieval-Service

import lakebase
print("lakebase import OK")

In [0]:
!python app.py